# The $\varepsilon$–$\delta$ game

**Math 140H — Honors Calculus I · Activity 2 companion notebook**

In the activity you play the *continuity game* by hand:

> Fix a point $x_0$. Your opponent hands you a target tolerance $\varepsilon > 0$.
> You must answer with a distance $\delta > 0$ so that
> $$|x - x_0| < \delta \quad\Longrightarrow\quad |f(x) - f(x_0)| < \varepsilon .$$
> If you can always answer, no matter how small $\varepsilon$ is, then $f$ is **continuous at $x_0$**.

This notebook lets you test your guesses. You

1. pick a function $f$ and a point $x_0$ (one short cell to edit — no Python needed),
2. choose $\varepsilon$ with a slider,
3. type in your $\delta$.

The notebook then works out **exactly how far $f$ moves** on the interval
$(x_0-\delta,\,x_0+\delta)$ — using symbolic computation (SymPy) when it can — and tells you
whether your $\delta$ wins the game for that $\varepsilon$.

The two functions from the activity are
$$
f(x) = \frac{x}{|x|}\ \ (x\neq 0), \qquad g(x) = \frac{1}{x^{2}}\ \ (x\neq 0),
$$
and both are continuous at $x_0 = 1$.


In [7]:
# --- Setup ---------------------------------------------------------------
# On a local Jupyter with the course environment, everything below is already
# installed and you can just run this cell.
#
# On Google Colab, uncomment the next line, run it once, then do
# Runtime > Restart session before continuing.
# %pip install -q sympy ipywidgets matplotlib numpy

# Colab needs this one line to show ipywidgets sliders:
try:
    import google.colab  # noqa: F401
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

%matplotlib inline


In [8]:
# --- The epsilon-delta game engine ------------------------------------------
# You do NOT need to read or understand this cell. Just run it once (Shift+Enter)
# and move on to the next one. It defines a single function, epsilon_delta_game(...).

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations,
    implicit_multiplication_application, convert_xor,
)

_X = sp.Symbol("x", real=True)
_TR = standard_transformations + (implicit_multiplication_application, convert_xor)

# colours
_C_CURVE, _C_EPS, _C_DEL, _C_RANGE, _C_PT = "#1f77b4", "#2ca02c", "#ff7f0e", "#d62728", "black"


def _parse_function(text):
    """Turn a string like '1/x^2' or 'x/abs(x)' into a SymPy expression in x."""
    expr = parse_expr(
        str(text), transformations=_TR,
        local_dict={"x": _X, "e": sp.E, "pi": sp.pi, "abs": sp.Abs},
    )
    stray = expr.free_symbols - {_X}
    if stray:
        raise ValueError(f"the function may only use the variable x; found {stray}")
    return expr


def _nice(v):
    """Best-effort tidy number: an integer or simple fraction if v is close to one."""
    v = float(v)
    if abs(v - round(v)) < 1e-9:
        return sp.Integer(round(v))
    fr = sp.Rational(v).limit_denominator(1000)
    if abs(float(fr) - v) < 1e-9:
        return fr
    return sp.Float(v, 6)


def _value_at(expr, x0):
    v = sp.simplify(expr.subs(_X, sp.nsimplify(x0)))
    if v.free_symbols or v in (sp.zoo, sp.nan, sp.oo, -sp.oo) or v.is_finite is False:
        raise ValueError(f"f(x0) is undefined at x0 = {x0}")
    return v


def _range_on(expr, a, b):
    """Range of expr over the closed interval [a, b].

    Returns dict with either {'unbounded': True} or
    {'lo', 'hi', 'exact': bool}.  'exact' means SymPy found it symbolically.
    """
    a_s, b_s = sp.nsimplify(a), sp.nsimplify(b)
    dom = sp.Interval(a_s, b_s)
    try:
        hi = sp.maximum(expr, _X, dom)
        lo = sp.minimum(expr, _X, dom)
        if hi in (sp.oo, sp.zoo) or lo in (-sp.oo, sp.zoo):
            return {"unbounded": True}
        if not (hi.free_symbols or lo.free_symbols):
            return {"lo": sp.nsimplify(lo), "hi": sp.nsimplify(hi), "exact": True}
    except Exception:
        pass
    # Numerical fallback (exact for step functions, a close estimate otherwise).
    fn = sp.lambdify(_X, expr, "numpy")
    xs = np.linspace(float(a), float(b), 20001)
    with np.errstate(all="ignore"):
        ys = np.asarray(fn(xs), dtype=float)
    ys = ys[np.isfinite(ys)]
    if ys.size == 0 or np.max(np.abs(ys)) > 1e6:
        return {"unbounded": True}
    return {"lo": _nice(ys.min()), "hi": _nice(ys.max()), "exact": False}


def _fmt(q):
    return sp.latex(sp.nsimplify(q)) if getattr(q, "is_number", False) else sp.latex(q)


def epsilon_delta_game(function_text, x0, eps_max=2.0, delta_start=0.5, view=1.5):
    """Interactive epsilon-delta game for f(x) = function_text at the point x0."""
    expr = _parse_function(function_text)
    x0f = float(x0)
    f0 = _value_at(expr, x0)
    f0f = float(f0)
    fn = sp.lambdify(_X, expr, "numpy")

    def _dev_numeric(delta):
        xs = np.linspace(x0f - delta, x0f + delta, 4001)
        with np.errstate(all="ignore"):
            ys = np.asarray(fn(xs), dtype=float)
        ys = ys[np.isfinite(ys)]
        if ys.size == 0:
            return np.inf
        return float(max(ys.max() - f0f, f0f - ys.min()))

    def _delta_that_works(eps):
        """Largest delta in (0, cap] whose worst deviation is <= eps (bisection)."""
        cap = max(delta_start, 1.0)
        if _dev_numeric(cap) <= eps:
            return cap, cap
        lo, hi = 0.0, cap
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            if _dev_numeric(mid) <= eps:
                lo = mid
            else:
                hi = mid
        return lo, cap

    # ---- widgets ---------------------------------------------------------
    eps_slider = widgets.FloatSlider(
        value=1.0, min=0.05, max=eps_max, step=0.05, description="ε  (target)",
        readout_format=".2f", continuous_update=False,
        style={"description_width": "80px"}, layout=widgets.Layout(width="440px"),
    )
    delta_box = widgets.FloatText(
        value=delta_start, step=0.05, description="δ  (your guess)",
        style={"description_width": "80px"}, layout=widgets.Layout(width="220px"),
    )
    feedback = widgets.HTMLMath()
    plot_out = widgets.Output()

    header = widgets.HTMLMath(
        value=(f"Testing continuity of $f(x) = {sp.latex(expr)}$ at "
               f"$x_0 = {sp.latex(sp.nsimplify(x0))}$, where $f(x_0) = {sp.latex(f0)}$.")
    )

    def refresh(*_):
        eps = float(eps_slider.value)
        delta = float(delta_box.value)
        plot_out.clear_output(wait=True)

        if delta <= 0:
            feedback.value = "<b>δ must be a positive number.</b>"
            with plot_out:
                _draw(eps, None, None)
            return

        rng = _range_on(expr, x0f - delta, x0f + delta)
        a_lbl = sp.latex(sp.nsimplify(x0f - delta))
        b_lbl = sp.latex(sp.nsimplify(x0f + delta))

        if rng.get("unbounded"):
            feedback.value = (
                f"On $({a_lbl},\\ {b_lbl})$ the values of $f$ are "
                f"<b>unbounded</b> (the interval reaches a pole or the "
                f"discontinuity). No $\\varepsilon$ can be met here &mdash; "
                f"choose a smaller $\\delta$."
            )
            with plot_out:
                _draw(eps, delta, None)
            return

        lo, hi = rng["lo"], rng["hi"]
        lo_f, hi_f = float(lo), float(hi)
        B = max(hi_f - f0f, f0f - lo_f)
        approx = "" if rng["exact"] else "\\approx "
        about = "" if rng["exact"] else "about "

        lines = [
            f"For $\\delta = {sp.latex(sp.nsimplify(delta))}$: on "
            f"$({a_lbl},\\ {b_lbl})$ the function stays in "
            f"$[{_fmt(lo)},\\ {_fmt(hi)}]$.",
            f"So for every $x$ with $|x - x_0| < \\delta$ we have "
            f"$|f(x) - f(x_0)| \\le {approx}{_fmt(_nice(B))}$ ({about}the tightest "
            f"$\\varepsilon$ this $\\delta$ guarantees).",
        ]

        if B <= eps + 1e-12:
            lines.append(
                f"<b style='color:{_C_EPS}'>&#10003; This &delta; wins the game for "
                f"&epsilon; = {eps:.2f}:</b> &nbsp; "
                f"$|x - x_0| < \\delta \\ \\Rightarrow\\ "
                f"|f(x) - f(x_0)| \\le {approx}{_fmt(_nice(B))} \\le \\varepsilon.$"
            )
        else:
            dw, cap = _delta_that_works(eps)
            if dw < 1e-6:
                tip = ("Even a tiny $\\delta$ leaves $|f(x)-f(x_0)|$ far from $0$ &mdash; "
                       "$f$ looks <b>discontinuous</b> at $x_0$.")
            else:
                tip = f"A $\\delta$ that does work for this $\\varepsilon$: $\\delta \\approx {dw:.4g}$."
            lines.append(
                f"<b style='color:{_C_RANGE}'>&#10007; This &delta; is too big for "
                f"&epsilon; = {eps:.2f}:</b> &nbsp; some $x$ within $\\delta$ of $x_0$ "
                f"has $|f(x) - f(x_0)| {approx}= {_fmt(_nice(B))} > \\varepsilon$. {tip}"
            )

        feedback.value = "<br>".join(lines)
        with plot_out:
            _draw(eps, delta, (lo_f, hi_f))

    def _draw(eps, delta, achieved):
        W = max(2.5 * (delta or 0.0), view)
        xs = np.linspace(x0f - W, x0f + W, 1400)
        with np.errstate(all="ignore"):
            ys = np.asarray(fn(xs), dtype=float)
        ys = np.where(np.isfinite(ys) & (np.abs(ys) < 1e4), ys, np.nan)

        fig, ax = plt.subplots(figsize=(6.4, 4.3))
        ax.plot(xs, ys, color=_C_CURVE, lw=2, zorder=3)
        ax.axhspan(f0f - eps, f0f + eps, color=_C_EPS, alpha=0.12, zorder=0,
                   label="ε-window (target)")
        for yv in (f0f - eps, f0f + eps):
            ax.axhline(yv, color=_C_EPS, lw=1, ls="--", zorder=1)
        if delta:
            ax.axvspan(x0f - delta, x0f + delta, color=_C_DEL, alpha=0.12, zorder=0,
                       label="δ-neighborhood (your guess)")
            for xv in (x0f - delta, x0f + delta):
                ax.axvline(xv, color=_C_DEL, lw=1, ls="--", zorder=1)
        if achieved is not None:
            lo_f, hi_f = achieved
            ax.hlines([lo_f, hi_f], x0f - (delta or 0), x0f + (delta or 0),
                      color=_C_RANGE, lw=2, zorder=4)
            ax.vlines(x0f - W * 0.98, lo_f, hi_f, color=_C_RANGE, lw=6, zorder=4,
                      label="range of f on the δ-neighborhood")
        ax.plot([x0f], [f0f], "o", color=_C_PT, ms=6, zorder=5)

        span = eps
        if achieved is not None:
            span = max(span, (achieved[1] - achieved[0]) / 2)
        pad = 1.5 * span + 0.4
        ax.set_ylim(f0f - pad, f0f + pad)
        ax.set_xlim(x0f - W, x0f + W)
        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.grid(alpha=0.25)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, fontsize=8,
                  frameon=False)
        fig.tight_layout()
        plt.show()

    eps_slider.observe(refresh, names="value")
    delta_box.observe(refresh, names="value")

    ui = widgets.VBox([
        header,
        widgets.HBox([eps_slider, delta_box]),
        feedback,
        plot_out,
    ])
    display(ui)
    refresh()


## Play

Edit the two values in the next cell, run it, then move the $\varepsilon$ slider and type a $\delta$.

- Powers: write `x^2` or `x**2`.
- Available: `abs(x)`, `sqrt(x)`, `sin(x)`, `cos(x)`, `exp(x)`, `log(x)`, `pi`, `e`.
- For the activity's $f(x) = x/|x|$, write `x/abs(x)` (same as `sign(x)` away from $0$).
- For the activity's $g(x) = 1/x^2$, write `1/x^2`.


In [9]:
function_text = "1/x^2"     # <-- your function of x
x0 = 1                      # <-- the point x0 where you test continuity

epsilon_delta_game(function_text, x0)


### The other activity function

In [5]:
epsilon_delta_game("x/abs(x)", x0=-2)


## What to try

1. **Start with $\varepsilon = 1$** at $x_0 = 1$, as in the activity. Find a $\delta$ that wins for
   $f(x) = 1/x^2$, then one for $f(x) = x/|x|$. Which function forces you to be more careful?
2. **Shrink $\varepsilon$ to $0.1$**, then smaller. Does your $\delta$ have to change? By how much?
   For $x/|x|$ near $x_0 = 1$ the function is *constant*, so one $\delta$ works for **every**
   $\varepsilon$ — see if the notebook agrees.
3. **Move $x_0$ to a discontinuity**: try `epsilon_delta_game("1/x^2", x0=0)` or
   `epsilon_delta_game("x/abs(x)", x0=0)`. Now shrink $\delta$ as far as you like — the guaranteed
   bound on $|f(x) - f(x_0)|$ stops improving. That stuck bound is exactly why the $\varepsilon$–$\delta$
   definition **fails** there.
4. **A steep continuous function**: try `epsilon_delta_game("x^3", x0=2)`. For a fixed $\varepsilon$,
   is the winning $\delta$ larger or smaller than for `x^3` at `x0=0`? Explain using the graph.
